# 07 - Additional Models: Logistic Regression, SVM, LSTM & Ensemble

**Member 1 — P. Chamika Janith Piyasena (CIT-24-01-0182)**  
**Group 6 — Spam Email/SMS Detection System**

This notebook trains **four additional models** on top of the Naive Bayes (NB05) and CNN (NB06) already built:

| # | Model | Type | Vectorization |
|---|-------|------|---------------|
| 1 | Logistic Regression | Classical ML | TF-IDF |
| 2 | Linear SVM | Classical ML | TF-IDF |
| 3 | LSTM (BiLSTM) | Deep Learning | Embedding |
| 4 | Voting Ensemble | Ensemble | NB + LR + SVM |

All models use the **same `full_pipeline()` from `pipeline.py`** so preprocessing is consistent with NB05 and CNN06.

---

## Setup
Run this first — handles both Google Colab and local VS Code environments.

In [ ]:
# ============================================================
# SETUP CELL - run this first, every time
# Works both locally (VS Code / Jupyter) and in Google Colab
# ============================================================
import os, sys, shutil

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Sandaru17513/NLP_Ctrl-Alt-Elite.git"
    REPO_DIR = "NLP_Ctrl-Alt-Elite"

    os.chdir('/content')
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.system(f"git clone {REPO_URL}")

    target_notebooks_dir = os.path.join('/content', REPO_DIR, 'notebooks')
    if os.path.exists(target_notebooks_dir):
        os.chdir(target_notebooks_dir)
    else:
        raise FileNotFoundError(f"Notebooks dir not found: {target_notebooks_dir}")

    os.system("pip install -q -r ../requirements.txt langdetect")
else:
    print("Running locally (VS Code / Jupyter). Using existing .venv environment.")

import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

sys.path.append(os.path.abspath("../src"))
print("IN_COLAB =", IN_COLAB)
print("Working directory:", os.getcwd())

## 0. Load & Preprocess Data

We load the raw CSV files and apply the project's `full_pipeline()` function — the same one used across all notebooks.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from pipeline import full_pipeline   # your src/pipeline.py

os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

# ── Load raw data ──────────────────────────────────────────────────────────────
df_train = pd.read_csv('../data/dataset.csv')
df_val   = pd.read_csv('../data/validation_dataset.csv')

print('Training shape:', df_train.shape)
print('Validation shape:', df_val.shape)

# ── Standardise column names (same logic as NB05 / CNN06) ─────────────────────
df_train = df_train.rename(columns={'text': 'email_text', 'label': 'email_label'})

df_val = df_val.rename(columns={'Email Text': 'email_text', 'Email Type': 'email_label'})
df_val['email_label'] = df_val['email_label'].map(
    {'Safe Email': 0, 'Phishing Email': 1}
)

print('Train label distribution:\n', df_train['email_label'].value_counts())
print('Val label distribution:\n',   df_val['email_label'].value_counts())

In [ ]:
# ── Apply full_pipeline() to every row ────────────────────────────────────────
# This mirrors the preprocessing in NB05 / CNN06 for a fair comparison.
print('Preprocessing training data (this takes ~1-2 min for 50k rows)...')
df_train['final_text'] = df_train['email_text'].apply(full_pipeline)

print('Preprocessing validation data...')
df_val['final_text'] = df_val['email_text'].apply(full_pipeline)

# ── Labels ────────────────────────────────────────────────────────────────────
# Training set: 'email_label' is already 0/1 (ham/spam)
X_all = df_train['final_text'].fillna('')
y_all = df_train['email_label'].astype(int)

X_val_raw = df_val['final_text'].fillna('')
y_val     = df_val['email_label'].astype(int)

print(f'Training corpus: {len(X_all)} rows')
print(f'Validation corpus: {len(X_val_raw)} rows')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f'X_train: {len(X_train)} | X_test: {len(X_test)}')
print(f'Spam ratio (train): {y_train.mean()*100:.1f}%')
print(f'Spam ratio (test): {y_test.mean()*100:.1f}%')

## 1. TF-IDF Vectorization (shared by LR + SVM)

TF-IDF weights rare, informative words more than common ones — better than raw Bag-of-Words for longer emails.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),      # unigrams + bigrams (captures "click here", "free prize")
    sublinear_tf=True,       # apply log(1+tf) — helps with skewed term frequencies
    min_df=2,                # ignore terms that appear in fewer than 2 docs
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
X_val_tfidf   = tfidf.transform(X_val_raw)

print('TF-IDF matrix shape (train):', X_train_tfidf.shape)
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
print('Saved: tfidf_vectorizer.pkl')

---
## 2. Model 1 — Logistic Regression

**Why this model?**  
Logistic Regression is a strong baseline for text classification. Combined with TF-IDF it is often competitive with complex deep learning models on spam detection while being fast and interpretable (you can inspect which tokens push toward spam).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
import matplotlib.pyplot as plt

lr_model = LogisticRegression(
    C=1.0,            # inverse regularization strength
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',   # handles ham/spam imbalance automatically
    random_state=42,
)

lr_model.fit(X_train_tfidf, y_train)
print('Logistic Regression trained!')

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
y_pred_lr  = lr_model.predict(X_test_tfidf)
y_prob_lr  = lr_model.predict_proba(X_test_tfidf)[:, 1]

print('=' * 50)
print(' LOGISTIC REGRESSION - TEST SET RESULTS')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}')
print(f' Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f' Recall:    {recall_score(y_test, y_pred_lr):.4f}')
print(f' F1-Score:  {f1_score(y_test, y_pred_lr):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Ham', 'Spam']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Ham', 'Spam']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Logistic Regression (Test Set)')
RocCurveDisplay.from_predictions(y_test, y_prob_lr, ax=axes[1], name='Logistic Regression')
axes[1].set_title('ROC Curve — Logistic Regression')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
plt.tight_layout()
plt.savefig('../reports/lr_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate on held-out validation set ───────────────────────────────────────
y_pred_lr_val = lr_model.predict(X_val_tfidf)
y_prob_lr_val = lr_model.predict_proba(X_val_tfidf)[:, 1]

print('=' * 50)
print(' LOGISTIC REGRESSION - VALIDATION SET RESULTS')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_val, y_pred_lr_val):.4f}')
print(f' Precision: {precision_score(y_val, y_pred_lr_val):.4f}')
print(f' Recall:    {recall_score(y_val, y_pred_lr_val):.4f}')
print(f' F1-Score:  {f1_score(y_val, y_pred_lr_val):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_val, y_prob_lr_val):.4f}')

joblib.dump(lr_model, '../models/logistic_regression_model.pkl')
print('Model saved: logistic_regression_model.pkl')

In [ ]:
# ── Inspect top features (great for viva!) ────────────────────────────────────
feature_names = np.array(tfidf.get_feature_names_out())
coef = lr_model.coef_[0]

top_spam_idx = np.argsort(coef)[-20:][::-1]
top_ham_idx  = np.argsort(coef)[:20]

print('Top 20 SPAM indicator words/phrases (Logistic Regression):')
for feat, score in zip(feature_names[top_spam_idx], coef[top_spam_idx]):
    print(f'  {feat:30s}  coef: {score:.3f}')

print()
print('Top 20 HAM indicator words/phrases (Logistic Regression):')
for feat, score in zip(feature_names[top_ham_idx], coef[top_ham_idx]):
    print(f'  {feat:30s}  coef: {score:.3f}')

---
## 3. Model 2 — Support Vector Machine (LinearSVC)

**Why this model?**  
SVMs maximise the margin between classes in high-dimensional TF-IDF space — historically the state-of-the-art for text classification. `LinearSVC` scales to large vocabulary sizes efficiently. It often beats Naive Bayes and even Logistic Regression on sparse text features.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# CalibratedClassifierCV wraps LinearSVC so we get probability estimates
# for ROC-AUC and ensemble voting later.
svm_base = LinearSVC(
    C=0.5,
    max_iter=2000,
    class_weight='balanced',
    random_state=42,
)
svm_model = CalibratedClassifierCV(svm_base, cv=3)
svm_model.fit(X_train_tfidf, y_train)
print('SVM trained!')

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
y_pred_svm = svm_model.predict(X_test_tfidf)
y_prob_svm = svm_model.predict_proba(X_test_tfidf)[:, 1]

print('=' * 50)
print(' LINEAR SVM - TEST SET RESULTS')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_test, y_pred_svm):.4f}')
print(f' Precision: {precision_score(y_test, y_pred_svm):.4f}')
print(f' Recall:    {recall_score(y_test, y_pred_svm):.4f}')
print(f' F1-Score:  {f1_score(y_test, y_pred_svm):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_test, y_prob_svm):.4f}')
print()
print(classification_report(y_test, y_pred_svm, target_names=['Ham', 'Spam']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_svm = confusion_matrix(y_test, y_pred_svm)
ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=['Ham', 'Spam']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Linear SVM (Test Set)')
RocCurveDisplay.from_predictions(y_test, y_prob_svm, ax=axes[1], name='Linear SVM')
axes[1].set_title('ROC Curve — Linear SVM')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
plt.tight_layout()
plt.savefig('../reports/svm_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate on validation set ────────────────────────────────────────────────
y_pred_svm_val = svm_model.predict(X_val_tfidf)
y_prob_svm_val = svm_model.predict_proba(X_val_tfidf)[:, 1]

print('=' * 50)
print(' LINEAR SVM - VALIDATION SET RESULTS')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_val, y_pred_svm_val):.4f}')
print(f' Precision: {precision_score(y_val, y_pred_svm_val):.4f}')
print(f' Recall:    {recall_score(y_val, y_pred_svm_val):.4f}')
print(f' F1-Score:  {f1_score(y_val, y_pred_svm_val):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_val, y_prob_svm_val):.4f}')

joblib.dump(svm_model, '../models/svm_model.pkl')
print('Model saved: svm_model.pkl')

---
## 4. Model 3 — Bidirectional LSTM

**Why this model?**  
LSTMs capture sequential context — word order matters for spam ("you have won" vs. "won you have" are semantically different). A Bidirectional LSTM reads the email both forward and backward, giving richer representations than the CNN's fixed window filters. This is your most expressive deep learning alternative.

Architecture:
```
Embedding → BiLSTM(128) → Dropout(0.5) → Dense(64, relu) → Dropout(0.3) → Dense(1, sigmoid)
```

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional, LSTM,
                                      Dense, Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np

# ── Hyperparameters (matched to CNN06 for fair comparison) ────────────────────
MAX_VOCAB = 15000
MAX_LEN   = 200
EMBED_DIM = 64

# ── Tokenize ──────────────────────────────────────────────────────────────────
lstm_tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
lstm_tokenizer.fit_on_texts(X_train)

X_train_seq = lstm_tokenizer.texts_to_sequences(X_train)
X_test_seq  = lstm_tokenizer.texts_to_sequences(X_test)
X_val_seq   = lstm_tokenizer.texts_to_sequences(X_val_raw)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post', truncating='post')

print(f'X_train_pad: {X_train_pad.shape}')
print(f'X_test_pad:  {X_test_pad.shape}')

joblib.dump(lstm_tokenizer, '../models/lstm_tokenizer.pkl')
print('Saved: lstm_tokenizer.pkl')

In [ ]:
# ── Build BiLSTM model ────────────────────────────────────────────────────────
lstm_model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=EMBED_DIM),

    Bidirectional(LSTM(128, return_sequences=False)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid'),
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

In [ ]:
# ── Class weights to handle ham/spam imbalance ────────────────────────────────
spam_count = int(y_train.sum())
ham_count  = int((y_train == 0).sum())
total      = len(y_train)

class_weights_lstm = {
    0: total / (2 * ham_count),
    1: total / (2 * spam_count),
}
print('Class weights:', class_weights_lstm)

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weights_lstm,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)
print('BiLSTM training complete!')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_lstm.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history_lstm.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('BiLSTM — Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_lstm.history['loss'],     label='Train Loss')
axes[1].plot(history_lstm.history['val_loss'], label='Val Loss')
axes[1].set_title('BiLSTM — Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
y_prob_lstm = lstm_model.predict(X_test_pad).flatten()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

print('=' * 50)
print(' BILSTM - TEST SET RESULTS')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_test, y_pred_lstm):.4f}')
print(f' Precision: {precision_score(y_test, y_pred_lstm):.4f}')
print(f' Recall:    {recall_score(y_test, y_pred_lstm):.4f}')
print(f' F1-Score:  {f1_score(y_test, y_pred_lstm):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_test, y_prob_lstm):.4f}')
print()
print(classification_report(y_test, y_pred_lstm, target_names=['Ham', 'Spam']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_lstm = confusion_matrix(y_test, y_pred_lstm)
ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=['Ham', 'Spam']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — BiLSTM (Test Set)')
RocCurveDisplay.from_predictions(y_test, y_prob_lstm, ax=axes[1], name='BiLSTM')
axes[1].set_title('ROC Curve — BiLSTM')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
plt.tight_layout()
plt.savefig('../reports/lstm_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate on validation set ────────────────────────────────────────────────
y_prob_lstm_val = lstm_model.predict(X_val_pad).flatten()
y_pred_lstm_val = (y_prob_lstm_val >= 0.5).astype(int)

print('=' * 50)
print(' BILSTM - VALIDATION SET RESULTS')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_val, y_pred_lstm_val):.4f}')
print(f' Precision: {precision_score(y_val, y_pred_lstm_val):.4f}')
print(f' Recall:    {recall_score(y_val, y_pred_lstm_val):.4f}')
print(f' F1-Score:  {f1_score(y_val, y_pred_lstm_val):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_val, y_prob_lstm_val):.4f}')

lstm_model.save('../models/bilstm_model.keras')
print('Model saved: bilstm_model.keras')

---
## 5. Model 4 — Soft-Voting Ensemble (NB + LR + SVM)

**Why this model?**  
Ensembles combine the strengths of multiple models. NB is great at flagging rare spam words; LR captures weighted feature combinations; SVM finds the maximal margin. By averaging their probability outputs (soft voting), errors in one model can be corrected by the other two — typically improving precision and recall together.

> **Note:** We reload the saved Naive Bayes model from NB05 so we don't have to re-train it.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# ── Reload Naive Bayes artifacts from NB05 ────────────────────────────────────
try:
    nb_model     = joblib.load('../models/naive_bayes_model.pkl')
    bow_vectorizer = joblib.load('../models/bow_vectorizer.pkl')
    print('Loaded NB model and BOW vectorizer from NB05.')
except FileNotFoundError:
    # If NB05 hasn't been run yet, train a quick NB here so the ensemble can proceed
    print('NB05 artifacts not found — training Naive Bayes here...')
    bow_vectorizer = CountVectorizer(max_features=5000)
    bow_vectorizer.fit(X_train)
    nb_model = MultinomialNB(alpha=0.1)
    nb_model.fit(bow_vectorizer.transform(X_train), y_train)
    joblib.dump(nb_model, '../models/naive_bayes_model.pkl')
    joblib.dump(bow_vectorizer, '../models/bow_vectorizer.pkl')
    print('NB model trained and saved.')

# ── BOW-transform all splits for NB ───────────────────────────────────────────
X_train_bow = bow_vectorizer.transform(X_train)
X_test_bow  = bow_vectorizer.transform(X_test)
X_val_bow   = bow_vectorizer.transform(X_val_raw)

In [ ]:
# ── Get probability predictions from all three models ─────────────────────────
p_nb  = nb_model.predict_proba(X_test_bow)[:, 1]
p_lr  = lr_model.predict_proba(X_test_tfidf)[:, 1]
p_svm = svm_model.predict_proba(X_test_tfidf)[:, 1]

# ── Soft vote: average the three probability estimates ────────────────────────
p_ensemble = (p_nb + p_lr + p_svm) / 3.0
y_pred_ens = (p_ensemble >= 0.5).astype(int)

print('=' * 50)
print(' SOFT-VOTING ENSEMBLE (NB + LR + SVM) - TEST SET')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_test, y_pred_ens):.4f}')
print(f' Precision: {precision_score(y_test, y_pred_ens):.4f}')
print(f' Recall:    {recall_score(y_test, y_pred_ens):.4f}')
print(f' F1-Score:  {f1_score(y_test, y_pred_ens):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_test, p_ensemble):.4f}')
print()
print(classification_report(y_test, y_pred_ens, target_names=['Ham', 'Spam']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_ens = confusion_matrix(y_test, y_pred_ens)
ConfusionMatrixDisplay(confusion_matrix=cm_ens, display_labels=['Ham', 'Spam']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Ensemble (Test Set)')
RocCurveDisplay.from_predictions(y_test, p_ensemble, ax=axes[1], name='Ensemble')
axes[1].set_title('ROC Curve — Ensemble')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
plt.tight_layout()
plt.savefig('../reports/ensemble_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluate ensemble on validation set ───────────────────────────────────────
p_nb_val  = nb_model.predict_proba(X_val_bow)[:, 1]
p_lr_val  = lr_model.predict_proba(X_val_tfidf)[:, 1]
p_svm_val = svm_model.predict_proba(X_val_tfidf)[:, 1]

p_ens_val = (p_nb_val + p_lr_val + p_svm_val) / 3.0
y_pred_ens_val = (p_ens_val >= 0.5).astype(int)

print('=' * 50)
print(' SOFT-VOTING ENSEMBLE - VALIDATION SET')
print('=' * 50)
print(f' Accuracy:  {accuracy_score(y_val, y_pred_ens_val):.4f}')
print(f' Precision: {precision_score(y_val, y_pred_ens_val):.4f}')
print(f' Recall:    {recall_score(y_val, y_pred_ens_val):.4f}')
print(f' F1-Score:  {f1_score(y_val, y_pred_ens_val):.4f}')
print(f' ROC-AUC:   {roc_auc_score(y_val, p_ens_val):.4f}')

---
## 6. Final Comparison — All Models

Side-by-side comparison of every model across both evaluation sets.

In [ ]:
# ── Collect all results ───────────────────────────────────────────────────────
# (Re-uses predictions computed above; assumes NB05 y_pred_nb / CNN06 y_pred_cnn
#  are NOT available here — we skip them and focus on this notebook's 4 models.
#  If you want to add NB05/CNN06 numbers, load their .pkl and run predict here.)

results = {
    'Logistic Regression': {
        'test':  (y_test, y_pred_lr,  y_prob_lr),
        'val':   (y_val,  y_pred_lr_val, y_prob_lr_val),
    },
    'Linear SVM': {
        'test':  (y_test, y_pred_svm,  y_prob_svm),
        'val':   (y_val,  y_pred_svm_val, y_prob_svm_val),
    },
    'BiLSTM': {
        'test':  (y_test, y_pred_lstm,  y_prob_lstm),
        'val':   (y_val,  y_pred_lstm_val, y_prob_lstm_val),
    },
    'Ensemble (NB+LR+SVM)': {
        'test':  (y_test, y_pred_ens,  p_ensemble),
        'val':   (y_val,  y_pred_ens_val, p_ens_val),
    },
}

rows = []
for model_name, splits in results.items():
    for split_name, (yt, yp, ypr) in splits.items():
        rows.append({
            'Model':    model_name,
            'Split':    split_name,
            'Accuracy':  round(accuracy_score(yt, yp), 4),
            'Precision': round(precision_score(yt, yp), 4),
            'Recall':    round(recall_score(yt, yp), 4),
            'F1-Score':  round(f1_score(yt, yp), 4),
            'ROC-AUC':   round(roc_auc_score(yt, ypr), 4),
        })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))
df_results.to_csv('../reports/all_models_comparison.csv', index=False)
print('\nSaved: all_models_comparison.csv')

In [ ]:
# ── Bar chart: F1-Score comparison on test set ────────────────────────────────
test_df = df_results[df_results['Split'] == 'test'].set_index('Model')

fig, ax = plt.subplots(figsize=(10, 5))
test_df[['F1-Score', 'ROC-AUC', 'Precision', 'Recall']].plot(
    kind='bar', ax=ax, rot=20, width=0.7
)
ax.set_title('Model Comparison — Test Set Metrics (NB07)')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Overlay ROC curves on one chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(y_test, y_prob_lr,    name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_svm,   name='Linear SVM',          ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_lstm,  name='BiLSTM',              ax=ax)
RocCurveDisplay.from_predictions(y_test, p_ensemble,   name='Ensemble',            ax=ax)

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_title('ROC Curve Comparison — All NB07 Models (Test Set)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/all_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| Model | Vectorization | Key Strength |
|-------|--------------|-------------|
| **Logistic Regression** | TF-IDF + bigrams | Fast, interpretable, strong baseline |
| **Linear SVM** | TF-IDF + bigrams | Best margin classifier for sparse text |
| **BiLSTM** | Learned Embedding | Captures word order and long-range context |
| **Ensemble (NB+LR+SVM)** | BOW + TF-IDF | Combines complementary model strengths |

**What to look for in the results:**
- A high **Recall** for spam is more important than Precision in spam detection (missing a spam is worse than a false alarm).
- Compare **Validation F1** across notebooks (NB05, CNN06, NB07) to pick the final production model.
- The Ensemble typically reduces variance; the BiLSTM typically excels when emails are long and contextual.

All model artifacts are saved in `../models/` and all evaluation plots are saved in `../reports/`.